This code aims to test and understand the catalog parser function.

Initial thought: Catalog parser streamlines multiple attributes in CIM objects, and creates a single clean structured input. This can improve the individual functions in the object builder library. It will also help in creating json files of outputs easily. 

In [1]:
from __future__ import annotations
import json
import logging


from cimgraph.databases import get_cim_profile 
from cimgraph.models import GraphModel

_log = logging.getLogger(__name__)


def catalog_parser(catalog_file, network):
    file = open(catalog_file)
    catalog = json.load(file)
    data = catalog['catalog']
    cim_profile, cim = get_cim_profile() # Import CIM profile
    obj = item_parser(data, network, cim)
    file.close()
    return obj

def item_parser(data:dict, network: GraphModel, cim):
    class_type = edge_class = eval(f'cim.{data["@type"]}')
    obj = class_type()
    network.add_to_graph(obj)

    for attribute in data:
        if type(data[attribute]) == str:
            if attribute in class_type.__dataclass_fields__:
                setattr(obj, attribute, data[attribute])
            else:
                _log.warning(f'Attribute {attribute} not found')
        elif type(data[attribute]) == list:
            if attribute in class_type.__dataclass_fields__:
                values = getattr(obj, attribute)
                for item in data[attribute]:
                    value = item_parser(item, network, cim)
                    values.append(value)
                setattr(obj, attribute, values)
    return obj


First part is simply initializing.

In [2]:
from __future__ import annotations
import json
import logging

from cimgraph.databases import get_cim_profile
from cimgraph.models import GraphModel

_log = logging.getLogger(__name__)


Now let us test each lines inside the catalog parser fuctions

In [3]:
import os
from cimgraph.databases import XMLFile
from cimgraph.models import FeederModel
import cimgraph.data_profile.cimhub_2023 as cim
os.environ['CIMG_CIM_PROFILE'] = 'cimhub_2023'
file = XMLFile(filename='../test_models/IEEE13.xml')
feeder = cim.Feeder(mRID= '49AD8E07-3BF9-A4E2-CB8F-C3722F837B62')
network = FeederModel(container=feeder, connection=file)

In [4]:
# file1 = open(file)
# catalog = json.load(filename='../test_models/hv69_12.json')

In [5]:
import json

# Open the JSON file
with open('../test_models/hv69_12.json', 'r') as file:
    # Load the JSON contents into the `catalog` variable
    catalog = json.load(file)

print(catalog)
print(catalog.keys())
print(catalog['catalog'])
print(catalog['catalog'].keys())
print(catalog['catalog'].values())

{'catalog': {'@type': 'PowerTransformer', 'name': 'hvmv69_12', 'vectorGroup': 'Yy', 'PowerTransformerEnd': [{'@type': 'PowerTransformerEnd', 'name': 'hvmv69_12_End_1', 'endNumber': '1', 'grounded': 'true', 'rground': '0', 'xground': '0', 'connectionKind': 'WindingConnection.Y', 'phaseAngleClock': '0', 'r': '1.594935', 'x': '18.90117', 'b': '0', 'ratedS': '20000000', 'ratedU': '69000'}, {'@type': 'PowerTransformerEnd', 'name': 'hvmv69_12_End_2', 'endNumber': '2', 'grounded': 'true', 'rground': '0', 'xground': '0', 'connectionKind': 'WindingConnection.Y', 'phaseAngleClock': '0', 'r': '0.052092802', 'x': '0', 'b': '0', 'ratedS': '20000000', 'ratedU': '12470'}]}}
dict_keys(['catalog'])
{'@type': 'PowerTransformer', 'name': 'hvmv69_12', 'vectorGroup': 'Yy', 'PowerTransformerEnd': [{'@type': 'PowerTransformerEnd', 'name': 'hvmv69_12_End_1', 'endNumber': '1', 'grounded': 'true', 'rground': '0', 'xground': '0', 'connectionKind': 'WindingConnection.Y', 'phaseAngleClock': '0', 'r': '1.594935', '

In [6]:
data = catalog['catalog']
cim_profile, cim = get_cim_profile() # Import CIM profile
print(cim_profile)

cimhub_2023


In [8]:
class_type = edge_class = eval(f'cim.{data["@type"]}')
obj = class_type()
network.add_to_graph(obj)


for attribute in data:
    if type(data[attribute]) == str:
        print(data[attribute])
        if attribute in class_type.__dataclass_fields__:
            setattr(obj, attribute, data[attribute])
            print(data[attribute])
        else:
            _log.warning(f'Attribute {attribute} not found')
            print('warning')
            print(attribute)
            print(data[attribute])
            print('\n')
    elif type(data[attribute]) == list:
        #print(data[attribute])
        if attribute in class_type.__dataclass_fields__:
                values = getattr(obj, attribute)
                for item in data[attribute]:
                    value = item_parser(item, network, cim)
                    values.append(value)
                setattr(obj, attribute, values)

print(obj)
#print(obj[0])


Attribute @type not found
Attribute @type not found
Attribute x not found
Attribute b not found
Attribute @type not found
Attribute x not found
Attribute b not found


PowerTransformer
warning
@type
PowerTransformer


hvmv69_12
hvmv69_12
Yy
Yy
{"@id": "e90b37e1-4024-4e87-8eec-1bf9ca5bd9e6", "@type": "PowerTransformer", "name": "hvmv69_12", "vectorGroup": "Yy", "PowerTransformerEnd": [{"@id": "a5fa648e-bebe-49c7-90c9-b89dc937a1d3", "@type": "PowerTransformerEnd"}, {"@id": "8ee9b67f-b1a9-4a32-ab92-438cff7dba98", "@type": "PowerTransformerEnd"}]}


In [34]:
print(obj)

{"@id": "0ac3d0ad-5dde-4a5c-9d5a-7be4225d9d27", "@type": "PowerTransformer", "name": "hvmv69_12", "vectorGroup": "Yy", "PowerTransformerEnd": [{"@id": "3d8e11a9-f99f-4e03-bf04-6738c2930b08", "@type": "PowerTransformerEnd"}, {"@id": "7abcfa9e-8539-4be7-86c6-64c022727c43", "@type": "PowerTransformerEnd"}]}


In [20]:
for attribute in data:
    print(attribute)

@type
name
vectorGroup
PowerTransformerEnd


In [9]:
data = catalog['catalog']
obj = item_parser(data, network, cim)

NameError: name 'item_parser' is not defined

In [13]:
import xmltodict
import json

# Input XML string
xml_data = """
<GridAtlas>
    <Task id="9">
        <Description>Enable End-Users to Assemble Grid Atlas Models</Description>
        <Tool>
            <Name>Object Builder</Name>
            <Components>
                <Component>
                    <Name>EasyCIM</Name>
                    <Purpose>Reduced form of each component</Purpose>
                </Component>
                <Component>
                    <Name>Catalog Parser</Name>
                    <Purpose>Parsing functionality</Purpose>
                </Component>
            </Components>
        </Tool>
    </Task>
</GridAtlas>
"""

file_xml_IEEE_13 = XMLFile(filename='../test_models/IEEE13.xml')
# Convert XML to a Python dictionary
data_dict = xmltodict.parse(file_xml_IEEE_13)

# Convert dictionary to JSON
json_data = json.dumps(data_dict, indent=4)

# Output JSON
print(json_data)

ModuleNotFoundError: No module named 'xmltodict'

In [9]:
from __future__ import annotations
import json
import logging

from cimgraph.databases import get_cim_profile
from cimgraph.models import GraphModel

_log = logging.getLogger(__name__)


def catalog_parser(catalog_file, network):
    file = open(catalog_file) ## opening json file
    catalog = json.load(file) ## loading jsonj file contents as catalog
    data = catalog['catalog'] ## catalog dictionary, first element (in this example, only single XF) saved into data
    cim_profile, cim = get_cim_profile() # Import CIM profile 
    obj = item_parser(data, network, cim) ## validating and simplifying json file objects into a single structure of obj. - needs modifications
    file.close()
    return obj

def item_parser(data:dict, network: GraphModel, cim):
    class_type = edge_class = eval(f'cim.{data["@type"]}')
    obj = class_type()
    network.add_to_graph(obj)

    for attribute in data:
        if type(data[attribute]) == str:
            if attribute in class_type.__dataclass_fields__:
                setattr(obj, attribute, data[attribute])
            else:
                _log.warning(f'Attribute {attribute} not found')
        elif type(data[attribute]) == list:
            if attribute in class_type.__dataclass_fields__:
                values = getattr(obj, attribute)
                for item in data[attribute]:
                    value = item_parser(item, network, cim)
                    values.append(value)
                setattr(obj, attribute, values)
    return obj

In [10]:
import os
from cimgraph.databases import XMLFile
from cimgraph.models import FeederModel
import cimgraph.data_profile.cimhub_2023 as cim
os.environ['CIMG_CIM_PROFILE'] = 'cimhub_2023'
file = XMLFile(filename='../test_models/IEEE13.xml')
feeder = cim.Feeder(mRID= '49AD8E07-3BF9-A4E2-CB8F-C3722F837B62')
network = FeederModel(container=feeder, connection=file)

Catalog_JSON_file_path = '../test_models/hv69_12.json'
Obj_output = catalog_parser(Catalog_JSON_file_path, network)

Attribute @type not found
Attribute @type not found
Attribute x not found
Attribute b not found
Attribute @type not found
Attribute x not found
Attribute b not found


In [25]:
print(Obj_output.PowerTransformerEnd[0]) ## .keys()

dict1 = vars(Obj_output.PowerTransformerEnd[0])
print(dict1.keys())
print(dict1.values())
# print(Obj_output[])

{"@id": "e991d35c-38a9-43a3-b874-f26930eff366", "@type": "PowerTransformerEnd", "name": "hvmv69_12_End_1", "endNumber": "1", "grounded": "true", "rground": "0", "xground": "0", "phaseAngleClock": "0", "connectionKind": "WindingConnection.Y", "r": "1.594935", "ratedS": "20000000", "ratedU": "69000"}
dict_keys(['identifier', 'mRID', 'aliasName', 'description', 'name', 'Names', 'endNumber', 'grounded', 'rground', 'xground', 'BaseVoltage', 'CoreAdmittance', 'FromMeshImpedance', 'PhaseTapChanger', 'RatioTapChanger', 'StarImpedance', 'Terminal', 'ToMeshImpedance', 'phaseAngleClock', 'connectionKind', 'r', 'ratedS', 'ratedU', 'PowerTransformer', '__uuid__', '__json_ld__'])
dict_values([UUID('e991d35c-38a9-43a3-b874-f26930eff366'), 'e991d35c-38a9-43a3-b874-f26930eff366', None, None, 'hvmv69_12_End_1', [], '1', 'true', '0', '0', None, None, [], None, None, None, None, [], '0', 'WindingConnection.Y', '1.594935', '20000000', '69000', None, <cimgraph.data_profile.identity.UUID_Meta object at 0x720